# CoAP: REST for constrained devices

GET, PUT, Observe and Block-wise over UDP — against a simulated greenhouse node on an in-memory network whose packet loss you choose.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.8.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.8.0-preview.1"

## A message on the wire
The RFC 7252 example: CON GET /temperature, 16 bytes.

In [ ]:
using IoTCom.Net.Protocols.Coap;

var get = new CoapMessage { Code = CoapCode.Get, MessageId = 0x7D34 };
get.UriPath = "/temperature";
var wire = get.Encode();
Console.WriteLine(Convert.ToHexString(wire));
foreach (var f in CoapAnatomy.Describe(wire)) Console.WriteLine($"{f.Name,-10} {f.Value}");

## A device and a client

In [ ]:
using System.Net;
using IoTCom.Net.Transports;

var net = new InMemoryDatagramNetwork();
var node = new IPEndPoint(IPAddress.Parse("10.0.0.40"), 5683);
await using var server = CoapServer.Create(o => o.UseInMemory(net, node));
await using var device = new CoapDeviceSimulator(server);
await server.StartAsync();

await using var coap = CoapClient.Create(o => o.UseInMemory(net).UseServer(node));
await coap.ConnectAsync();
foreach (var link in await coap.DiscoverAsync()) Console.WriteLine($"{link.Path,-22} {link.ResourceType} {(link.Observable ? "(observable)" : "")}");
Console.WriteLine((await coap.GetAsync("/sensors/temperature")).PayloadText);
Console.WriteLine((await coap.GetAsync("/sensors/temperature", CoapContentFormat.SenMLJson)).PayloadText);

## Observe
The device pushes a notification when the value changes.

In [ ]:
using var cts = new CancellationTokenSource();
var count = 0;
var watch = Task.Run(async () => { await foreach (var n in coap.ObserveAsync("/sensors/soil", ct: cts.Token)) { Console.WriteLine($"soil {n.PayloadText} (obs {n.ObserveSequence})"); if (++count == 3) break; } });
await coap.PutAsync("/actuators/valve", "80");
while (!watch.IsCompleted) await device.StepAsync();

## A lossy link
30 % of datagrams vanish; confirmable messages are retransmitted until acknowledged.

In [ ]:
net.LossRate = 0.3;
coap.Options.Transmission.AckTimeout = TimeSpan.FromMilliseconds(50);
coap.Options.Transmission.MaxRetransmit = 8;
var log = await coap.GetAsync("/device/log");   // Block2: several exchanges
Console.WriteLine($"{log.Payload.Length} bytes; {coap.Statistics.RetransmissionCount} retransmissions; {net.Dropped} datagrams lost");

## Going further
The Gallery demo *Smart greenhouse over CoAP* draws every datagram on a message sequence chart. See `docs/en/protocols/coap.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*